# Ask CIP | Validate first, train once

Keep this notebook private. With the current pilot bundle, select **CPU**, **Python 3**, **Latest**, and leave **High-RAM off**. Select **Run all** and upload `cip-colab-inputs.zip`. This checks the data and downloads a report on CPU. It uses no paid AI APIs.

Use **T4 GPU** with standard RAM for one epoch only after an approved bundle passes preflight. Choose L4 if T4 is unavailable. The allowance depends on the runtime assigned and your Colab balance. The notebook skips installs, model downloads and training when evidence is missing.

A valid GPU run downloads `cip-candidate.zip` with the ONNX, tokenizer, calibration and paired report. Keep the shipped reader until the independent CIP tests validate the candidate.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

from google.colab import files
work = Path("/content")
script = work / "ask_cip_train.py"
cases = work / "cases.jsonl"
baseline = work / "ask-reader"
output = work / "cip-candidate"
model_files = ("model_optimized.onnx", "tokenizer.json", "calibration.json")
required = [script, cases, *(baseline / name for name in model_files)]
provenance = work / "dataset-report.json"

if any(not path.is_file() for path in required):
    print("Choose cip-colab-inputs.zip.")
    uploaded = files.upload()
    archive = uploaded.get("cip-colab-inputs.zip")
    if archive is None:
        raise ValueError("Choose the local file named cip-colab-inputs.zip and run this cell again.")
    with ZipFile(BytesIO(archive)) as bundle:
        missing = [str(path.relative_to(work)) for path in required if str(path.relative_to(work)) not in bundle.namelist()]
        if missing:
            raise ValueError("Missing in input ZIP: " + ", ".join(missing))
        baseline.mkdir(exist_ok=True)
        for path in required + [provenance]:
            name = str(path.relative_to(work))
            if name in bundle.namelist():
                path.write_bytes(bundle.read(name))
    del archive, uploaded
print("Training cases and shipped baseline are ready.")

In [ ]:
import json
import subprocess
import sys

preflight_report = work / "preflight-report.json"
check = subprocess.run([sys.executable, str(script), "preflight", "--cases", str(cases),
                        "--baseline", str(baseline), "--output", str(preflight_report)])
if check.returncode not in (0, 2) or not preflight_report.is_file():
    raise RuntimeError(f"Preflight could not run (exit {check.returncode}).")
evidence = json.loads(preflight_report.read_text())
ready = check.returncode == 0 and evidence["ready"]
if not ready:
    print("Training paused. Keep CPU selected. Read the blocking reasons in the report.")

In [ ]:
import os

trained = False
if ready:
    import torch

    if not torch.cuda.is_available():
        print("Data is ready. Select T4 GPU before the training run.")
    else:
        print("GPU:", torch.cuda.get_device_name(0))
        if output.exists():
            print("The output folder already exists. It will be checked in the download cell.")
        else:
            subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers==4.57.3",
                            "onnx==1.20.0", "onnxruntime==1.23.2"], check=True)
            subprocess.run([sys.executable, str(script), "train", "--cases", str(cases),
                            "--baseline", str(baseline), "--output", str(output), "--epochs", "1"],
                           check=True, env={**os.environ, "HF_HUB_DISABLE_TELEMETRY": "1"})
        paired = output / "paired-report.json"
        if not paired.is_file() or json.loads(paired.read_text())["cases_sha256"] != evidence["cases_sha256"]:
            raise ValueError("The existing candidate is incomplete or was trained on other cases. Keep these files and use a fresh runtime.")
        details = json.loads((output / "preflight-report.json").read_text())
        if details["baseline_artifacts"] != evidence["baseline_artifacts"]:
            raise ValueError("The existing candidate was compared to different baseline artifacts. Use a fresh runtime.")
        trained = True

In [ ]:
from zipfile import ZIP_DEFLATED

delivery = preflight_report
if trained:
    print(json.dumps(json.loads(paired.read_text()), indent=2))
    delivery = work / "cip-candidate.zip"
    with ZipFile(delivery, "w", compression=ZIP_DEFLATED) as bundle:
        for path in [paired, preflight_report, *(output / name for name in model_files)]:
            bundle.write(path, f"{output.name}/{path.name}")
        if provenance.is_file():
            bundle.write(provenance, f"{output.name}/{provenance.name}")
    with ZipFile(delivery) as bundle:
        bad = bundle.testzip()
        if bad:
            raise ValueError(f"Download ZIP failed its integrity check: {bad}")
print(f"Downloading {delivery.name}. Verify the download before disconnecting the runtime.")
files.download(str(delivery))

The browser downloads the report when the run stops at preflight. A successful training run downloads the candidate package. Both remain accessible through Colab’s Files panel. After the download finishes, use **Runtime → Disconnect and delete runtime**.